In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
print(df.isnull().sum())

categorical = df.select_dtypes(include='object').columns.tolist()
numerical = [c for c in df.select_dtypes(exclude='object').columns if c != 'converted']
print('\ncategorical:', categorical)
print('numerical  :', numerical)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
print('\nmissing after fill:', df.isnull().sum().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical  : ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

missing after fill: 0


In [3]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [7]:
print('Mode for the column industry :-', df['industry'].mode()[0])

Mode for the column industry :- technology


In [8]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [9]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f'{a:26s} & {b:18s}: {df[a].corr(df[b]):.4f}')

interaction_count          & lead_score        : 0.9157
number_of_courses_viewed   & lead_score        : 0.7572
number_of_courses_viewed   & interaction_count : 0.7216
annual_income              & interaction_count : 0.1228


In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

# target must not be in the dataframes
del df_train['converted']
del df_val['converted']
del df_test['converted']

print(len(df_train), len(df_val), len(df_test))

3000 1000 1000


In [11]:
def mi(col):
    return round(mutual_info_score(df_train[col], y_train), 2)

options = ['industry', 'location', 'lead_source', 'employment_status']
mi_scores = pd.Series({c: mi(c) for c in options}).sort_values(ascending=False)
print(mi_scores)
print('\n Mutual info score:', mi_scores.index[0])

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64

 Mutual info score: lead_source


In [12]:
def train_and_eval(cat, num, C=1.0):
    cols = cat + num
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[cols].to_dict(orient='records'))
    X_val = dv.transform(df_val[cols].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

base_acc = train_and_eval(categorical, numerical)
print('accuracy (rounded):', round(base_acc, 2), '| unrounded:', base_acc)

accuracy (rounded): 0.65 | unrounded: 0.645


In [13]:
features = categorical + numerical
results = []
for f in features:
    cat = [c for c in categorical if c != f]
    num = [n for n in numerical if n != f]
    acc = train_and_eval(cat, num)
    results.append((f, acc, base_acc - acc))

elim = pd.DataFrame(results, columns=['feature_removed', 'accuracy', 'difference']).set_index('feature_removed')
elim

,accuracy,difference
feature_removed,,
lead_source,0.642,0.003
industry,0.645,0.000
employment_status,0.645,0.000
location,0.645,0.000
annual_income,0.724,-0.079
number_of_courses_viewed,0.643,0.002
interaction_count,0.601,0.044
lead_score,0.644,0.001


In [15]:
options = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
sub = elim.loc[options]
print(sub)
print('\n smallest difference:', sub['difference'].idxmin())

                          accuracy  difference
feature_removed                               
lead_source                  0.642       0.003
number_of_courses_viewed     0.643       0.002
interaction_count            0.601       0.044

 smallest difference: number_of_courses_viewed


In [16]:
rows = []
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    rows.append((C, round(train_and_eval(categorical, numerical, C=C), 3)))
res6 = pd.DataFrame(rows, columns=['C', 'val_accuracy'])
print(res6)

best = res6['val_accuracy'].max()
print('\n smallest C with best accuracy:', res6[res6.val_accuracy == best]['C'].min())

          C  val_accuracy
0  0.000001         0.598
1  0.000010         0.598
2  0.000100         0.613
3  0.001000         0.645

 smallest C with best accuracy: 0.001
